# Data Model JSON Parser
by Machine Data Insights Inc.

## Instructions
1. Download and unzip the latest 'Splunk Common Information Model' from Splunkbase (<https://splunkbase.splunk.com/app/1621>).
2. Create a 'models' folder in the same directory as this notebook.
3. Copy the data model .json files (Alerts.json, Performance.json, etc.) from the unzipped /Splunk_SA_CIM/default/data/models folder to the /models folder created in step 2.
4. OPTIONAL: Update the filename in the csv_path = "./Splunk_Data_Model_Objects_Fields_604.csv" line near the bottom of this code to reflect the Splunk CIM app version the data model .json files were obtained from. You can, of course, name this file anything you'd like.

When you Shift-Enter to run this code the output file will be located in the same folder as this Notebook file (unless you change the path).  

In [1]:
import os
import json
import pandas as pd

# Load the data model JSON data for each file in the /models folder
directory = './models'

records = []
prefix = "BaseEvent."
prefix2 = "BaseSearch."

for filename in os.listdir(directory):
    file_path = os.path.join(directory, filename)
    if os.path.isfile(file_path):
        
        with open(file_path, "r") as f:
            data_model_data = json.load(f)
        
        # Extract the object-level info
        model = data_model_data["modelName"]
        for obj in data_model_data["objects"]:
            object_name = obj.get("objectName")
            parent_name = obj.get("parentName", "")
            base_search = obj.get("baseSearch", "")
            owner = f"{parent_name}.{object_name}" if parent_name else object_name

            # print(type(obj.get("constraints", [])))
            # print(obj.get("constraints", []))

            # Process fields
            for field in obj.get("fields", []):
                records.append({
                    "model": model,
                    "object_name": object_name,
                    "parentName": parent_name,
                    "owner": owner,
                    "dataset": owner[len(prefix):] if owner.startswith(prefix) else owner[len(prefix2):] if owner.startswith(prefix2) else owner,
                    "field_name": field.get("fieldName"),
                    "display_name": field.get("displayName"),
                    "type": field.get("type"),
                    "required": field.get("required", False),
                    "recommended": field.get("comment", {}).get("recommended", False),
                    "extracted_type": "extracted",
                    "calculation_type": "",
                    "expression": "",
                    "base_search": base_search,
                    "constraints": "; ".join(c.get("search", "") for c in obj.get("constraints", []) if isinstance(c, dict) and "search" in c),
                    "prescribed_values": ", ".join(field.get("comment", {}).get("expected_values", [])) if field.get("comment", {}).get("expected_values") else "",
                    "description": field.get("comment", {}).get("description", "")
                })
        
            # Process calculated fields
            for calc in obj.get("calculations", []):
                calc_type = calc.get("calculationType")
                expression = calc.get("expression")
                for field in calc.get("outputFields", []):
                    records.append({
                        "model": model,
                        "object_name": object_name,
                        "parentName": parent_name,
                        "owner": owner,
                        "dataset": owner[len(prefix):] if owner.startswith(prefix) else owner[len(prefix2):] if owner.startswith(prefix2) else owner,
                        "field_name": field.get("fieldName"),
                        "display_name": field.get("displayName"),
                        "type": field.get("type"),
                        "required": field.get("required", False),
                        "recommended": field.get("comment", {}).get("recommended", False),
                        "extracted_type": "calculated",
                        "calculation_type": calc_type,
                        "expression": expression,
                        "base_search": base_search,                        
                        "constraints": "; ".join(c.get("search", "") for c in obj.get("constraints", []) if isinstance(c, dict) and "search" in c),
                        "prescribed_values": ", ".join(field.get("comment", {}).get("expected_values", [])) if field.get("comment", {}).get("expected_values") else "",
                        "description": field.get("comment", {}).get("description", "")
                    })

# Convert to DataFrame and export
df = pd.DataFrame(records)

# FOR VERSION CONTROL: Update the filename below to reflect the version of the Splunk Common Information Model JSON files used.
# For example: If the SPLUNK CIM app version is 6.0.4, the filename is "Splunk_Data_Model_Objects_Fields_604.csv"
csv_path = "./splunk_data_model_objects_fields_604.csv"

df.to_csv(csv_path, index=False)
csv_path


'./splunk_data_model_objects_fields_604.csv'